# Market-data ETL — task runner

Pick one or more **tasks** and edit their **config section** below, then **Run All**.

**Tasks available**
- `md_snapshot_zip_to_parquet` — decompress daily snapshot zips into per-ticker parquet
  (`<OUT>/<yyyy>/<YYYYMMDD>/<ticker>.parquet`)
- `md_trade_zip_to_parquet` — decompress daily trade zips into per-ticker parquet
  (`<OUT>/<yyyy>/<YYYYMMDD>/<ticker>.parquet`)
- `md_eod_build` — distil the per-ticker snapshot + trade parquet into **daily EOD bars
  and cross-day features**: open/high/low/close, vwap, volume, amount, preclose,
  `trading_minutes` (distinct minutes with an orderbook update), forward returns
  (`fwd_ret_1d/5d/21d`), and `adv63` (trailing 63-day mean of `amount`). Writes a cached
  intermediate panel (`<OUT>/daily/<yyyy>/<YYYYMMDD>.parquet`) and one final file per day
  (`<OUT>/processed/<YYYYMMDD>.csv`, all tickers). Default `<OUT>` is
  `/data/equity_data/hkg/md_eod`.
- `hkg_earnings_scrape` — scrape HK-listed company earnings + announcement dates and
  partition them by announcement date (`<OUT>/<YYYYMMDD>.csv`, default
  `/data/equity_data/hkg/earnings/scrape`). This task **scrapes from the network**
  (akshare / HKEXnews) rather than reading local zips, so it has no `src`.

Each task has its own section in the `TASKS` config dict; set `RUN_TASKS` to the list
of tasks to run (in order). The actual work runs in this repo's dedicated venv (`.venv`)
via the tested `tasks/<task>.py` script, so it always uses the right deps no matter which
kernel this notebook is running on.

Every task declares a **`kind`** that selects how the runner drives it:
- `zip_days` — discover days by globbing `<src>/*.zip`, pass `--src` + `--date-range`.
- `eod_build` — discover trading days from the snapshot tree; the task auto-builds the
  lookback (63d) + forward (21d) window needed for `adv63` / forward returns, so the
  runner only forwards the date range. Edge days get NaN for those windowed features.
- `scrape` — no local input; pass `--out`/`--work-dir`, and `--date-range` (from
  `START`/`END`) selects which announcement-date partition files are (re)written.

> Safety: `DRY_RUN = True` by default — it only prints the command that *would* run.
> Set `DRY_RUN = False` to actually run. For `zip_days`, re-running a day overwrites its
> parquet; for `eod_build`, cached `daily/` bars are reused (pass nothing to rebuild —
> the task only rebuilds missing days unless run with `--force`) and `processed/` CSVs are
> rewritten; for `scrape`, re-running rewrites the in-range partition files (and re-uses
> already-scraped companies cached under the task's `work_dir`).

In [ ]:
# ============================ CONFIG — edit me ============================
from pathlib import Path

# --- Shared settings — apply to every task in RUN_TASKS ------------------
RUN_TASKS = [                            # tasks to run, in order (see TASKS below)
    # "md_snapshot_zip_to_parquet",
    # "md_trade_zip_to_parquet",
    "md_eod_build",                    # <- uncomment to build daily EOD bars + features
    # "hkg_earnings_scrape",             # <- uncomment to scrape HK earnings
]
MARKET  = "hkg"          # market code           (--market)
ASSET   = "eq"           # asset class           (--asset; eq = equity)
START   = "20250101"     # first day YYYYMMDD, or None = earliest available
END     = "20260430"     # last  day YYYYMMDD, or None = latest available
WORKERS = None           # None = script default (min(20, cpus)); or set an int
DRY_RUN = True           # True = just list the days; False = actually convert

REPO    = Path.cwd()                                  # this repo
VENV_PY = str(REPO / ".venv" / "bin" / "python")

# --- Per-task config — one section per task ------------------------------
# `kind` selects how the runner drives the task:
#   "zip_days"  — glob <src>/*.zip for days, pass --src + --date-range.
#   "scrape"    — no local input; pass --out/--work-dir + --date-range.
#   "eod_build" — discover days from the snapshot tree; build EOD bars + features.
TASKS = {
    # md_snapshot_zip_to_parquet: daily snapshot zips -> per-ticker parquet
    "md_snapshot_zip_to_parquet": {
        "kind":   "zip_days",
        "script": str(REPO / "tasks" / "md_snapshot_zip_to_parquet.py"),
        "src":    "/home/wangfc/tmp_data",  # root of <YYYYMM>/<YYYYMMDD>.zip files
        "out":    None,      # None = derive from market/asset (.../md_snapshot/raw)
    },
    # md_trade_zip_to_parquet: daily trade zips -> per-ticker parquet
    "md_trade_zip_to_parquet": {
        "kind":   "zip_days",
        "script": str(REPO / "tasks" / "md_trade_zip_to_parquet.py"),
        "src":    "/home/wangfc/md_trade",  # root of <YYYYMM>/<YYYYMMDD>.zip files
        "out":    None,      # None = derive from market/asset (.../md_trade/raw)
    },
    # md_eod_build: tick (snapshot + trade) -> daily EOD bars + features -> per-day CSV.
    # Discovers trading days from the snapshot tree (no zips). For each target day it
    # auto-builds an extra lookback (63d) + forward (21d) window of bars so adv63 and
    # the forward returns are populated; edge days get NaN for those.
    "md_eod_build": {
        "kind":          "eod_build",
        "script":        str(REPO / "tasks" / "md_eod_build.py"),
        "snapshot_root": None,   # None = derive from market/asset (.../md_snapshot/raw)
        "trade_root":    None,   # None = derive from market/asset (.../md_trade/raw)
        "out":           None,   # None = derive (.../md_eod; writes daily/ + processed/)
        "cross_check":   False,  # True = cross-check vol vs trade ticks (diagnostic, slower)
    },
    # hkg_earnings_scrape: scrape HK earnings + announcement dates -> per-day partitions.
    # No local input — it scrapes akshare/HKEXnews. START/END select which
    # announcement-date partition files get (re)written (None/None = all days).
    "hkg_earnings_scrape": {
        "kind":     "scrape",
        "script":   str(REPO / "tasks" / "hkg_earnings_scrape.py"),
        "out":      None,    # None = derive from market/asset (.../earnings/scrape)
        "work_dir": None,    # None = <out_parent>/scrape_staging (intermediate + resume files)
    },
}
# =========================================================================

assert RUN_TASKS, "RUN_TASKS is empty — list at least one task to run."
for _t in RUN_TASKS:
    assert _t in TASKS, f"Unknown task: {_t!r} (known: {sorted(TASKS)})"

In [ ]:
# Discover the available days for one task and apply the shared date range.
from pathlib import Path

def discover_days(cfg):
    """Return (available, lo, hi, days) for a task config.

    Globs <src>/*.zip and <src>/*/*.zip (per-month folders), extracts the
    YYYYMMDD stems, and filters to the inclusive shared START..END range
    (None on either side = earliest/latest available).
    """
    src = cfg["src"]
    available = sorted({p.stem for p in Path(src).glob("*.zip")}
                       | {p.stem for p in Path(src).glob("*/*.zip")})
    assert available, f"No *.zip found under {src}"

    lo = START or available[0]
    hi = END   or available[-1]
    days = [d for d in available if lo <= d <= hi]
    return available, lo, hi, days

In [ ]:
# Run each selected task, streaming progress live. Dispatches on the task's `kind`.
import subprocess


def _run_cmd(cmd):
    """Print the command; run it (unless DRY_RUN), streaming output + exit code."""
    print("\nCommand:\n ", " ".join(cmd), "\n")
    if DRY_RUN:
        print("DRY_RUN = True — nothing executed. Set DRY_RUN = False to run.")
        return
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    print(f"Exit code: {rc}", "OK" if rc == 0 else "FAILURES — see output above")


def _date_range_arg():
    """Shared START/END -> a --date-range value, or None when both are open."""
    if not START and not END:
        return None
    return f"{START or ''}:{END or ''}"


def run_zip_days(task, cfg):
    available, lo, hi, days = discover_days(cfg)
    print(f"Source        : {cfg['src']}")
    print(f"Available zips: {len(available)}  ({available[0]} .. {available[-1]})")
    print(f"Range         : {lo} .. {hi}")
    print(f"Days selected : {len(days)}")
    for d in days:
        print("   ", d)
    if not days:
        print("No days match the selected range — skipping.")
        return
    cmd = [VENV_PY, cfg["script"],
           "--market", MARKET, "--asset", ASSET,
           "--date-range", f"{lo}:{hi}", "--src", cfg["src"]]
    if cfg.get("out"):
        cmd += ["--out", cfg["out"]]
    if WORKERS:
        cmd += ["--workers", str(WORKERS)]
    _run_cmd(cmd)


def run_scrape(task, cfg):
    dr = _date_range_arg()
    print(f"Input         : (none — scrapes akshare / HKEXnews)")
    print(f"Output        : {cfg.get('out') or 'derived from market/asset'}")
    print(f"Partitions    : {dr or 'ALL announcement dates'}")
    cmd = [VENV_PY, cfg["script"], "--market", MARKET, "--asset", ASSET]
    if dr:
        cmd += ["--date-range", dr]
    if cfg.get("out"):
        cmd += ["--out", cfg["out"]]
    if cfg.get("work_dir"):
        cmd += ["--work-dir", cfg["work_dir"]]
    if WORKERS:
        cmd += ["--workers", str(WORKERS)]
    _run_cmd(cmd)


def run_eod_build(task, cfg):
    # The task discovers trading days from the snapshot tree itself and auto-builds
    # the lookback/forward window, so the notebook only forwards the date range.
    dr = _date_range_arg()
    print(f"Snapshot root : {cfg.get('snapshot_root') or 'derived (.../md_snapshot/raw)'}")
    print(f"Trade root    : {cfg.get('trade_root') or 'derived (.../md_trade/raw)'}")
    print(f"Output        : {cfg.get('out') or 'derived (.../md_eod -> daily/ + processed/)'}")
    print(f"Target days   : {dr or 'ALL available days'}")
    print(f"Cross-check   : {bool(cfg.get('cross_check'))}")
    cmd = [VENV_PY, cfg["script"], "--market", MARKET, "--asset", ASSET]
    if dr:
        cmd += ["--date-range", dr]
    if cfg.get("snapshot_root"):
        cmd += ["--snapshot-root", cfg["snapshot_root"]]
    if cfg.get("trade_root"):
        cmd += ["--trade-root", cfg["trade_root"]]
    if cfg.get("out"):
        cmd += ["--out", cfg["out"]]
    if cfg.get("cross_check"):
        cmd += ["--cross-check"]
    if WORKERS:
        cmd += ["--workers", str(WORKERS)]
    _run_cmd(cmd)


RUNNERS = {"zip_days": run_zip_days, "scrape": run_scrape, "eod_build": run_eod_build}


def run_task(task):
    cfg = TASKS[task]
    kind = cfg.get("kind", "zip_days")
    print("=" * 72)
    print(f"Task          : {task}  (kind={kind})")
    runner = RUNNERS.get(kind)
    if runner is None:
        print(f"Unknown kind {kind!r} — known: {sorted(RUNNERS)}. Skipping.")
        return
    runner(task, cfg)


for _task in RUN_TASKS:
    run_task(_task)